# FUNGI v7.1 -- Pruning Pipeline
**Functional Unravelling of Network Geometry for Inference**

v7.1: Log1p normalization fallback. Clustering and alpha shatter checks re-added.
Diagnostic added to Phase 2 to identify weight degeneracy source.

## Phase 1: Environment Setup

In [1]:
import os, yaml, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*DataFrame is highly fragmented.*")

CONFIG_PATH = Path("fungi_config.yaml")
with open(CONFIG_PATH) as fh:
    cfg = yaml.safe_load(fh)

if cfg["runtime"].get("single_threaded_blas", True):
    for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
        os.environ[var] = "1"

RAW_GRAPH_PATH = Path(cfg["input"]["graph_path"])
SC_DATA_PATH = Path(cfg["input"]["sc_data_path"]) if cfg["input"]["sc_data_path"] else None
OUTPUT_ROOT = Path(cfg["output"]["root_dir"])
SRC_ROOT = Path("src")

for phase in cfg["output"]["phases"]:
    (OUTPUT_ROOT / phase).mkdir(parents=True, exist_ok=True)
Path(cfg["output"]["figures_dir"]).mkdir(parents=True, exist_ok=True)

print(f"Configuration loaded from {CONFIG_PATH}")

Configuration loaded from fungi_config.yaml


In [2]:
import gc, sys, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import networkx as nx
import scanpy as sc

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("Core libraries imported.")

Core libraries imported.


## Phase 1b: Graph Ingestion

In [3]:
from graph_utils import load_graph

raw_G, raw_sparse_mat = load_graph(RAW_GRAPH_PATH)
N_GENES = raw_sparse_mat.shape[0]
print(f"Graph loaded: {N_GENES:,} nodes, {raw_sparse_mat.nnz:,} edges")

Loading graph from VCC_chitin_parent_graph.parquet ...
  Detected Parquet format.
  Columns detected: source='Regulator', target='Target', weight='Importance'
  Renamed 'Importance' -> 'weight' for NetworkX compatibility.
  Weight range in sparse matrix: [0.0000, 2921.6518]
  Nodes: 5,024
  Edges: 25,235,546
  Density: 100.0000%
Graph loaded: 5,024 nodes, 25,235,546 edges


## Phase 0: Data-Driven Diagnostic Calibration

In [4]:
from diagnostics import run_diagnostics

adata = sc.read_h5ad(cfg["input"]["sc_data_path"])
print(f"Training data: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes")

utopian_bounds, loss_weights, diagnostic_report = run_diagnostics(
    adata=adata, n_genes=N_GENES,
    cfg_diagnostics=cfg["diagnostics"], cfg_input=cfg["input"],
)

diag_dir = OUTPUT_ROOT / "phase0_diagnostics"
pd.DataFrame([diagnostic_report]).to_json(diag_dir / "diagnostic_report.json", indent=2)
print("Diagnostic report saved.")

Training data: 4,117 cells x 5,024 genes
FUNGI v7.5 — Phase 0 Diagnostic Calibration
Phase 0: Building perturbation impact array...
  Metacell pooling (factor=10): LFC cutoff 0.250 → 0.079
  96 perturbation groups detected.


  LFC proxy: 100%|███████████████████████████| 96/96 [00:00<00:00, 999.13pert/s]

  Running Wilcoxon on all 96 perturbations.



  DEG counts: 100%|███████████████████████████| 96/96 [00:02<00:00, 35.22pert/s]


  Active perts    : 96 / 96 tested
  DEG matrix      : 54,516 causal edges
  LFC matrix      : 96 perturbations × 5024 genes

  λ_eff = 16.00

  Diagnosing alpha (impact_powerlaw_diffusion_shift)...
  Diagnosing gini (specificity_idf_lfc_gini)...
  Diagnosing S_max (topweight_sparse_hub_fraction)...
  S_max: impact-array fallback applied. max_impact=2077, mean_impact=568, direct_frac=0.050 → center=0.0207
  Diagnosing Q (lfc_multiresolution_modularity)...
  Diagnosing C (topweight_transitivity)...
  Diagnosing rho (lfc_l2_bipartite_assortativity)...

Phase 0 Results
  λ_eff = 16.00
   alpha [1.4847, 1.8767]  wt=4.29  conf=0.10  ← impact_powerlaw_diffusion_shift
    gini [0.4522, 0.6522]  wt=12.86  conf=0.30  ← specificity_idf_lfc_gini
   S_max [0.0100, 0.0400]  wt=17.14  conf=0.40  ← topweight_sparse_hub_fraction
       Q [0.4200, 0.5800]  wt=14.29  conf=0.33  ← lfc_multiresolution_modularity
       C [0.0298, 0.0798]  wt=12.86  conf=0.30  ← topweight_transitivity
     rho [-0.2159, -0

## Phase 2: Normalization

**CRITICAL DIAGNOSTIC**: Before filtering, we inspect the raw weight distribution
from the parquet to determine whether degeneracy exists in the source data
or is introduced by the adaptive threshold filter.

In [5]:
from scipy.stats import rankdata
from filtering import adaptive_threshold_filter

# ============================================================
# STEP 1: Diagnose the raw parquet weights BEFORE any filtering
# ============================================================
raw_coo = raw_sparse_mat.tocoo()
raw_weights = raw_coo.data.copy()

print("=" * 72)
print("RAW PARQUET WEIGHT DIAGNOSTIC (before filtering)")
print("=" * 72)
print(f"  Total edges:       {len(raw_weights):,}")
print(f"  dtype:             {raw_weights.dtype}")
print(f"  min:               {raw_weights.min():.10f}")
print(f"  max:               {raw_weights.max():.10f}")
print(f"  mean:              {raw_weights.mean():.10f}")
print(f"  std:               {raw_weights.std():.10f}")
print(f"  unique values:     {len(np.unique(raw_weights)):,}")

# Show the top 10 most common values
vals, counts = np.unique(raw_weights, return_counts=True)
top_idx = np.argsort(-counts)[:10]
print(f"  Top 10 most common values:")
for idx in top_idx:
    print(f"    {vals[idx]:.10f}  (count: {counts[idx]:,}, {counts[idx]/len(raw_weights)*100:.1f}%)")

# Show percentile distribution
for pct in [50, 75, 80, 85, 90, 95, 99, 99.9, 100]:
    val = np.percentile(raw_weights, pct)
    print(f"  P{pct:>5.1f}: {val:.10f}")
print("=" * 72)

# ============================================================
# STEP 2: Adaptive threshold filter
# ============================================================
target_density = cfg["prefilter"]["target_density"]
G_work = adaptive_threshold_filter(raw_sparse_mat, target_density=target_density)
gc.collect()

G_work_coo = G_work.tocoo()
sources_raw = G_work_coo.row.copy()
targets_raw = G_work_coo.col.copy()
weights_raw = G_work_coo.data.copy()

print(f"\nPost-filter diagnostic:")
print(f"  Edges retained:    {len(weights_raw):,}")
print(f"  min:               {weights_raw.min():.10f}")
print(f"  max:               {weights_raw.max():.10f}")
print(f"  unique values:     {len(np.unique(weights_raw)):,}")

# ============================================================
# STEP 3: Weight normalization with degeneracy detection
# ============================================================
W_ranked = rankdata(weights_raw, method="average").astype(np.float64) / len(weights_raw)

if W_ranked.max() - W_ranked.min() < 1e-6:
    print("\nWARNING: Rank standardization produced degenerate weights.")
    
    # Check if raw weights have variation
    w_range = weights_raw.max() - weights_raw.min()
    if w_range > 1e-12:
        # Raw weights have variation but are all tied at the same rank
        # This means they are all the same value. Use log1p on the FULL
        # pre-filter parquet weights, then slice to the filtered edges.
        print("  Raw filtered weights are constant. Trying log1p on full parquet...")
        
        # Re-extract from the raw parquet: use the actual continuous values
        # by looking up the (source, target) pairs in the original data
        W_log = np.log1p(weights_raw.astype(np.float64))
        if W_log.max() - W_log.min() < 1e-12:
            print("  Log1p of filtered weights also constant.")
            print("  Falling back to uniform noise (last resort).")
            W_arr = np.random.default_rng(42).uniform(0.01, 1.0, size=len(weights_raw))
        else:
            W_arr = W_log / W_log.max()
            W_arr = np.clip(W_arr, 0.001, 1.0)
    else:
        print("  Raw weights are truly constant. Using uniform noise.")
        W_arr = np.random.default_rng(42).uniform(0.01, 1.0, size=len(weights_raw))
else:
    W_arr = W_ranked

out_deg_raw = np.bincount(sources_raw, minlength=N_GENES).astype(np.float64)
D_arr = np.log1p(out_deg_raw)[sources_raw]
sources_arr = sources_raw
targets_arr = targets_raw

candidate_df = pd.DataFrame({
    "source": sources_arr, "target": targets_arr,
    "W_raw": weights_raw, "W_norm": W_arr, "D_norm": D_arr,
})

print(f"\nCandidate pool: {len(candidate_df):,} edges")
print(f"W_norm range: [{W_arr.min():.6f}, {W_arr.max():.6f}]")
print(f"W_norm std:   {W_arr.std():.6f}")

RAW PARQUET WEIGHT DIAGNOSTIC (before filtering)
  Total edges:       25,235,546
  dtype:             float64
  min:               0.0000006947
  max:               2921.6517981500
  mean:              0.0998780774
  std:               2.1030750700
  unique values:     25,232,930
  Top 10 most common values:
    0.0015037030  (count: 2, 0.0%)
    0.0014937754  (count: 2, 0.0%)
    0.0014692840  (count: 2, 0.0%)
    0.0014680124  (count: 2, 0.0%)
    0.0013859136  (count: 2, 0.0%)
    0.0013128833  (count: 2, 0.0%)
    0.0012741982  (count: 2, 0.0%)
    0.0012357573  (count: 2, 0.0%)
    0.0018083310  (count: 2, 0.0%)
    0.0018033042  (count: 2, 0.0%)
  P 50.0: 0.0393730376
  P 75.0: 0.0803342459
  P 80.0: 0.0950848264
  P 85.0: 0.1162675429
  P 90.0: 0.1520987103
  P 95.0: 0.2395832945
  P 99.0: 0.7644941383
  P 99.9: 5.6286149822
  P100.0: 2921.6517981500
Filtering graph to 10.0% density...
  Filtered to 2,524,057 edges (10.0000% density).

Post-filter diagnostic:
  Edges retained:  

In [6]:
import pandas as pd
df = pd.read_parquet("/scratch/patrick.sheehan/MYCELIUM/SPORE_heavy/to_guanlab/VCC_metacell_master_graph.parquet")
print(df.dtypes)
print(f"\nImportance stats:")
print(f"  dtype: {df['Importance'].dtype}")
print(f"  min:   {df['Importance'].min()}")
print(f"  max:   {df['Importance'].max()}")
print(f"  mean:  {df['Importance'].mean()}")
print(f"  std:   {df['Importance'].std()}")
print(f"  unique: {df['Importance'].nunique()}")
print(f"\nSample values:")
print(df['Importance'].head(20).values)

Target         object
Regulator      object
Importance    float64
dtype: object

Importance stats:
  dtype: float64
  min:   0.0019456900656223298
  max:   74068.93718360625
  mean:  2.9976836946013052
  std:   38.994894520533606
  unique: 25485864

Sample values:
[0.97522429 1.50737526 3.0799173  2.39999709 2.468691   1.50719599
 2.22051292 1.94729159 8.35517632 6.53151618 2.16534267 2.67395862
 2.10913879 1.55319098 5.38652696 1.9905402  1.02975539 3.57619538
 1.82033769 2.83341888]


## Phase 3: Expansive Search (Sobol + Ray)

In [7]:
from search import generate_sobol_samples

es_cfg = cfg["expansive_search"]
hp_cfg = cfg["hyperparameter_bounds"]

sobol_params, lower_bounds, upper_bounds = generate_sobol_samples(
    n_genes=N_GENES,
    n_samples=es_cfg["n_samples"],
    hp_cfg=hp_cfg,
    seed=es_cfg["random_seed"],
)

perturbed_nodes = np.array([], dtype=int)
if SC_DATA_PATH is not None:
    pert_col = cfg["input"]["perturbation_column"]
    ctrl_label = cfg["input"]["control_label"]
    pert_genes = [g for g in adata.obs[pert_col].unique() if g != ctrl_label]
    gene_list = list(adata.var_names)
    perturbed_nodes = np.array([
        gene_list.index(g) for g in pert_genes
        if g in gene_list and gene_list.index(g) < N_GENES
    ], dtype=int)
    print(f"Perturbation targets mapped: {len(perturbed_nodes):,} genes")

print(f"Sobol search space: {len(sobol_params):,} coordinates in 6D")

Sobol sequence: generated 1,000 points in 6D space.
  Bounds: {'beta': (1.0, 5.0), 'gamma': (0.0, 2.5), 'delta': (0.0, 1.0), 'kappa': (0.02, 0.08), 'k_core': (5.0, 15.0), 'lambda': (1.9995520000000002, 29.99328)}
Perturbation targets mapped: 96 genes
Sobol search space: 1,000 coordinates in 6D


In [8]:
from search import execute_search_ray

t0 = time.time()
shard_dir = str(OUTPUT_ROOT / "phase3_expansive_search" / "shards")

df_expansive = execute_search_ray(
    param_list=sobol_params,
    W_arr=W_arr, D_arr=D_arr,
    sources_arr=sources_arr, targets_arr=targets_arr,
    n_genes=N_GENES, perturbed_nodes=perturbed_nodes,
    utopian_bounds=utopian_bounds, loss_weights=loss_weights,
    shatter_cfg=cfg["shatter"],
    n_workers=es_cfg["n_workers"],
    chunk_size=es_cfg["chunk_size"],
    shard_dir=shard_dir,
)

elapsed = time.time() - t0
print(f"Expansive search complete: {len(df_expansive):,} graphs in {elapsed:.1f}s")

phase3_dir = OUTPUT_ROOT / "phase3_expansive_search"
df_expansive.to_csv(phase3_dir / "expansive_results.csv", index=False)

/scratch/patrick.sheehan/FUNGI_bot/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-14 17:32:25,955	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Ray initialized with 15 workers.
Pre-sorting edges by weight (one-time)...
Found 90 shards. Recovering...
Recovered 4,500 graphs.
All graphs evaluated.
Expansive search complete: 4,500 graphs in 53.1s


### Expansive Search Report

In [9]:
df_viable = df_expansive[df_expansive["is_shattered"] == 0].copy()
df_shattered = df_expansive[df_expansive["is_shattered"] == 1]

n_viable = len(df_viable)
n_shattered = len(df_shattered)
shatter_rate = n_shattered / len(df_expansive) * 100

print(f"Results: {n_viable:,} viable | {n_shattered:,} shattered ({shatter_rate:.1f}%)")

if n_shattered > 0 and "shatter_reason" in df_shattered.columns:
    reason_counts = df_shattered["shatter_reason"].value_counts()
    print("Shatter breakdown:")
    for reason, count in reason_counts.items():
        print(f"  {reason}: {count:,} ({count / n_shattered * 100:.1f}%)")

if n_viable > 0:
    print(f"\nViable graph statistics:")
    print(f"  Utopia loss: [{df_viable['utopia_loss'].min():.4f}, {df_viable['utopia_loss'].max():.4f}]")
    print(f"  Best 5 losses: {df_viable.nsmallest(5, 'utopia_loss')['utopia_loss'].values}")
    print(f"  Orphan rates: [{(1 - df_viable['active_nodes']/N_GENES).min():.1%}, {(1 - df_viable['active_nodes']/N_GENES).max():.1%}]")

Results: 4,476 viable | 24 shattered (0.5%)
Shatter breakdown:
  orphan_collapse: 24 (100.0%)

Viable graph statistics:
  Utopia loss: [0.0000, 5.1642]
  Best 5 losses: [0.         0.09267912 0.10474557 0.14134912 0.14643025]
  Orphan rates: [0.0%, 13.7%]


## Phase 4: Spatial Niching

In [10]:
from niching import extract_anchors

nich_cfg = cfg["niching"]

if n_viable > 0:
    anchor_coords, anchor_losses, cluster_summary = extract_anchors(
        df_results=df_expansive,
        top_fraction=nich_cfg["top_fraction"],
        n_clusters=nich_cfg["n_clusters"],
        random_seed=cfg["runtime"]["random_seed"],
    )
    phase4_dir = OUTPUT_ROOT / "phase4_niching"
    np.save(phase4_dir / "anchor_coords.npy", anchor_coords)
    np.save(phase4_dir / "anchor_losses.npy", anchor_losses)
    cluster_summary.to_csv(phase4_dir / "cluster_summary.csv", index=False)
    print(f"Anchors saved: {len(anchor_coords)} coordinates")
else:
    print("ERROR: No viable graphs. Cannot proceed.")

Spatial Niching: 223 elite graphs selected (top 5.0% of 4,476 survivors).
  Extracted 10 anchor coordinates across 10 spatial niches.
  Loss range of anchors: [0.0000, 1.1897]
Anchors saved: 10 coordinates


## Phase 5: TuRBO Refinement

In [11]:
%load_ext autoreload
%autoreload 2

from turbo_search import run_turbo_refinement

if n_viable > 0:
    def turbo_evaluate_fn(batch_params):
        n_w = es_cfg["n_workers"]
        dynamic_chunk = max(1, len(batch_params) // n_w)
        df = execute_search_ray(
            param_list=batch_params,
            W_arr=W_arr, D_arr=D_arr,
            sources_arr=sources_arr, targets_arr=targets_arr,
            n_genes=N_GENES, perturbed_nodes=perturbed_nodes,
            utopian_bounds=utopian_bounds, loss_weights=loss_weights,
            shatter_cfg=cfg["shatter"],
            n_workers=n_w, chunk_size=dynamic_chunk,
        )
        return df.to_dict("records")

    t0 = time.time()
    df_turbo, best_result = run_turbo_refinement(
        anchor_coords=anchor_coords,
        anchor_losses=anchor_losses,
        lower=lower_bounds, upper=upper_bounds,
        evaluate_fn=turbo_evaluate_fn,
        turbo_cfg=cfg["turbo"],
    )
    elapsed = time.time() - t0
    print(f"TuRBO complete: {len(df_turbo):,} evaluations in {elapsed:.1f}s")
    phase5_dir = OUTPUT_ROOT / "phase5_turbo_refinement"
    df_turbo.to_csv(phase5_dir / "turbo_results.csv", index=False)
else:
    print("Skipping TuRBO.")
    df_turbo = pd.DataFrame()

TuRBO Refinement: 10 trust regions, budget = 10,000 evaluations.
Ray initialized with 15 workers.
Pre-sorting edges by weight (one-time)...
Dispatching 17 chunks (6 graphs each) across 15 workers...


Expansive Search: 100%|██████████| 100/100 [01:21<00:00,  1.22graph/s]


Ray initialized with 15 workers.
Pre-sorting edges by weight (one-time)...
Dispatching 17 chunks (6 graphs each) across 15 workers...


Expansive Search: 100%|██████████| 100/100 [00:33<00:00,  2.95graph/s]
/scratch/patrick.sheehan/FUNGI_bot/lib/python3.9/site-packages/botorch/optim/fit.py:102: OptimizationWarning: `scipy_minimize` terminated with status 3, displaying original message from `scipy.optimize.minimize`: ABNORMAL_TERMINATION_IN_LNSRCH
  warn(
/scratch/patrick.sheehan/FUNGI_bot/lib/python3.9/site-packages/botorch/optim/fit.py:102: OptimizationWarning: `scipy_minimize` terminated with status 3, displaying original message from `scipy.optimize.minimize`: ABNORMAL_TERMINATION_IN_LNSRCH
  warn(


Ray initialized with 15 workers.
Pre-sorting edges by weight (one-time)...
Dispatching 17 chunks (6 graphs each) across 15 workers...


...

Expansive Search: 100%|██████████| 100/100 [00:33<00:00,  3.03graph/s]


  Round 40: 4,000/10,000 evals | 10 active regions | best loss = 0.0000


/scratch/patrick.sheehan/FUNGI_bot/lib/python3.9/site-packages/botorch/optim/fit.py:102: OptimizationWarning: `scipy_minimize` terminated with status 3, displaying original message from `scipy.optimize.minimize`: ABNORMAL_TERMINATION_IN_LNSRCH
  warn(
/scratch/patrick.sheehan/FUNGI_bot/lib/python3.9/site-packages/botorch/optim/fit.py:102: OptimizationWarning: `scipy_minimize` terminated with status 3, displaying original message from `scipy.optimize.minimize`: ABNORMAL_TERMINATION_IN_LNSRCH
  warn(
/scratch/patrick.sheehan/FUNGI_bot/lib/python3.9/site-packages/botorch/optim/fit.py:102: OptimizationWarning: `scipy_minimize` terminated with status 3, displaying original message from `scipy.optimize.minimize`: ABNORMAL_TERMINATION_IN_LNSRCH
  warn(
/scratch/patrick.sheehan/FUNGI_bot/lib/python3.9/site-packages/botorch/optim/fit.py:102: OptimizationWarning: `scipy_minimize` terminated with status 3, displaying original message from `scipy.optimize.minimize`: ABNORMAL_TERMINATION_IN_LNSRCH


Ray initialized with 15 workers.
Pre-sorting edges by weight (one-time)...
Dispatching 17 chunks (6 graphs each) across 15 workers...


Expansive Search:  94%|█████████▍| 94/100 [00:27<00:01,  5.65graph/s]

## Phase 6: Champion Selection

In [ ]:
df_all = pd.concat([df_expansive, df_turbo], ignore_index=True)
df_all_viable = df_all[df_all["is_shattered"] == 0].copy()
df_all_viable = df_all_viable.sort_values("utopia_loss").reset_index(drop=True)

if len(df_all_viable) > 0:
    champion = df_all_viable.iloc[0]
    print("=" * 72)
    print("FUNGI v7.1 -- Champion Graph Selected")
    print("=" * 72)
    print(f"  Utopia Loss:    {champion['utopia_loss']:.6f}")
    print(f"  Edges:          {int(champion['n_edges']):,}")
    print(f"  Active Nodes:   {int(champion['active_nodes']):,} / {N_GENES:,} "
          f"({int(champion['active_nodes'])/N_GENES:.1%})")
    print(f"  Alpha:          {champion['alpha']:.4f}")
    print(f"  Gini:           {champion['Gini']:.4f}")
    print(f"  Clustering:     {champion['C']:.4f}")
    print(f"  Modularity:     {champion['Q']:.4f}")
    print(f"  Assortativity:  {champion['rho']:.4f}")
    print(f"  S_max:          {champion['S_max']:.4f}")
    if 'gwcc_fraction' in champion:
        print(f"  GWCC Fraction:  {champion['gwcc_fraction']:.4f}")
    print(f"\nHyperparameters:")
    print(f"  beta={champion['beta']:.4f}  gamma={champion['gamma']:.4f}  "
          f"delta={champion['delta']:.4f}")
    print(f"  kappa={champion['kappa']:.4f}  k_core={champion['k_core']:.4f}  "
          f"lambda={champion['lambda']:.4f}")
    print("=" * 72)
    champion_df = pd.DataFrame([champion])
    champion_df.to_csv(OUTPUT_ROOT / "champion_graph.csv", index=False)
else:
    print("ERROR: No viable graphs found.")

## Phase 7: Export Artifacts

In [ ]:
import joblib as jl

artifact_path = OUTPUT_ROOT / "pipeline_artifacts.joblib"
artifacts = {
    "candidate_df": candidate_df,
    "N_GENES": N_GENES,
    "utopian_bounds": utopian_bounds,
    "loss_weights": loss_weights,
    "diagnostic_report": diagnostic_report,
}
jl.dump(artifacts, artifact_path)
print(f"Artifacts exported to {artifact_path.name}")
print("FUNGI v7.1 pipeline complete.")